# Step 3. Adaptation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/3_adaptation.ipynb)

A backup for the tutorial: the same step without an assistant. Run the cells from top to bottom.

In [ ]:
%pip install -q sweetpea pandas

**The question.** Do people adjust to how frequent conflict is?

**Prediction.** The congruency effect is smaller in mostly incongruent blocks than in mostly congruent blocks.

**The alternative to rule out.** In a mostly congruent block the word predicts the ink, so people can simply learn to follow the word.

**The analysis.** The congruency effect in mostly congruent blocks against the congruency effect in mostly incongruent blocks.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

### The proportion is a weight

The third argument of a level is its weight: 3 to 1 gives three quarters.

In [ ]:
WEIGHTS = {"mostly congruent": (3, 1), "mostly incongruent": (1, 3)}     # (congruent, incongruent)


def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word


def block(context):
    congruent_weight, incongruent_weight = WEIGHTS[context]
    congruency = Factor("congruency", [
        DerivedLevel("congruent", WithinTrial(same, [colour, word]), congruent_weight),
        DerivedLevel("incongruent", WithinTrial(different, [colour, word]), incongruent_weight),
    ])
    return CrossBlock(design=[colour, word, congruency], crossing=[colour, congruency],
                      constraints=[MinimumTrials(24)])

### A session of four blocks

Each block is generated on its own.

In [ ]:
blocks = []
for number, context in enumerate(["mostly congruent", "mostly incongruent", "mostly incongruent", "mostly congruent"], start=1):
    trials = pd.DataFrame(synthesize_trials(block(context), 1, CMSGen)[0])
    trials.insert(0, "context", context)
    trials.insert(0, "block", number)
    blocks.append(trials)
session = pd.concat(blocks, ignore_index=True)
print(pd.crosstab(session["context"], session["congruency"]))

### How well does the word tell the ink?

For each word, the share of trials on which it is shown in its own colour. That number is the alternative explanation.

In [ ]:
session["word_is_ink"] = session["word"] == session["colour"]
print(session.groupby(["context", "word"])["word_is_ink"].mean().round(2).unstack())

**Check.** Are the proportions what was asked for? How many incongruent trials does a mostly congruent block have: is that enough? The word was left out of the crossing: does every word predict its ink equally well?

**Try.** Change the weights to `(4, 1)` and `(1, 4)`. What happens to the rare trials?